# 3. Classification: Logistic Regression, KNN, Naive Bayes + ROC Curve
Works on **any dataset** (binary or multi-class).

**Steps:** Pick file -> Prepare data -> Logistic Regression -> KNN (choose k) -> Naive Bayes -> Compare models -> ROC curves -> Best threshold

> A **file explorer window** opens in Step 2, then you type the **target column** name.

## Step 0: Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report,
                             roc_curve, roc_auc_score, auc)

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## Step 1: CONFIG (defaults are fine)

In [ ]:
TEST_SIZE      = 0.2
RANDOM_STATE   = 42
CV_FOLDS       = 5
K_MAX          = 20          # KNN: try k = 1 ... K_MAX
CLASS_WEIGHT   = None        # None or 'balanced' (for imbalanced data, used in Logistic Regression)
POSITIVE_CLASS = None        # binary only: name of the "positive" class, e.g. 'Yes'. None = automatic
DROP_COLS      = []          # columns to drop manually, e.g. ['id', 'name']

## Step 2: Select Dataset from File Explorer

In [ ]:
def pick_file():
    """Opens the OS file explorer. Falls back to typing the path if no GUI is available."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        path = filedialog.askopenfilename(
            title='Select your dataset',
            filetypes=[('Data files', '*.csv *.xlsx *.xls *.json *.txt *.tsv'), ('All files', '*.*')])
        root.destroy()
    except Exception:
        print('File dialog not available -> type the path instead.')
        path = input('Enter full file path: ')
    return path.strip().strip('"').strip("'")

def load_data(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    if ext == '.json':
        return pd.read_json(path)
    try:
        return pd.read_csv(path, sep=None, engine='python')
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=None, engine='python', encoding='latin-1')

file_path = pick_file()
if not file_path:
    raise SystemExit('No file selected. Re-run this cell.')

df = load_data(file_path)
df.columns = df.columns.astype(str).str.strip()
print('Loaded:', file_path, '| Shape:', df.shape)
df.head()

In [ ]:
print('Columns:', list(df.columns))
TARGET = input('Enter the TARGET (class) column name: ').strip()
while TARGET not in df.columns:
    TARGET = input(f"'{TARGET}' not found. Enter a valid column name: ").strip()
print('TARGET =', TARGET)

## Step 3: Prepare Data (clean, split, impute, encode, scale)

In [ ]:
# ---- basic cleaning ----
def text_cols(d):
    return [col for col in d.columns
            if pd.api.types.is_object_dtype(d[col]) or pd.api.types.is_string_dtype(d[col])]

df = df.drop_duplicates().reset_index(drop=True)
for col in text_cols(df):
    df[col] = df[col].where(df[col].isna(), df[col].astype(str).str.strip())
df = df.replace(['?', 'NA', 'N/A', 'na', 'n/a', 'NaN', 'nan', 'null', 'NULL', 'None', '-', '--', ''], np.nan)

for col in text_cols(df):                       # numbers stored as text -> numeric
    if col == TARGET:
        continue
    conv = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() > 0 and conv.notna().sum() >= 0.9 * df[col].notna().sum():
        df[col] = conv
for col in df.select_dtypes(include='bool').columns:
    if col != TARGET:
        df[col] = df[col].astype(int)

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

# ---- drop useless columns ----
to_drop = [col for col in DROP_COLS if col in df.columns]
for col in df.columns:
    if col == TARGET:
        continue
    if (df[col].isnull().mean() > 0.5 or df[col].nunique(dropna=True) <= 1
            or (col in text_cols(df) and df[col].nunique() == len(df))
            or pd.api.types.is_datetime64_any_dtype(df[col])):
        to_drop.append(col)
to_drop = list(dict.fromkeys(to_drop))
print('Dropped columns:', to_drop if to_drop else 'None')
df = df.drop(columns=to_drop)
print('Shape:', df.shape)

In [ ]:
# ---- encode target (classes -> 0,1,2,...) ----
y_text = df[TARGET].astype(str)
class_names = sorted(y_text.unique())
if len(class_names) == 2 and POSITIVE_CLASS is not None and str(POSITIVE_CLASS) in class_names:
    class_names = [c for c in class_names if c != str(POSITIVE_CLASS)] + [str(POSITIVE_CLASS)]
class_map = {name: i for i, name in enumerate(class_names)}
y = y_text.map(class_map)
n_classes = len(class_names)
AVG = 'binary' if n_classes == 2 else 'weighted'

print('Class mapping:', class_map)
if n_classes == 2:
    print(f"Binary classification -> positive class (1) = '{class_names[1]}'")
else:
    print(f'Multi-class classification with {n_classes} classes')
if n_classes > 20:
    print('WARNING: too many unique target values - this looks like a regression problem, not classification.')

# class distribution
counts = y_text.value_counts()
display(pd.DataFrame({'count': counts, '%': (counts / len(y_text) * 100).round(2)}))
plt.figure(figsize=(6, 3.5))
sns.countplot(x=y_text, order=class_names, palette='Set2')
plt.title(f'Class distribution of {TARGET}')
plt.show()

In [ ]:
# ---- train/test split FIRST (avoids data leakage), then preprocess ----
X = df.drop(columns=TARGET)
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)

num_c = X.select_dtypes(include=np.number).columns.tolist()
cat_c = [col for col in X.columns if col not in num_c]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                      ('scale',  StandardScaler())]), num_c),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_c)
])

X_train_p = preprocessor.fit_transform(X_train)      # fit on train only
X_test_p  = preprocessor.transform(X_test)
feature_names = [n.split('__', 1)[1] for n in preprocessor.get_feature_names_out()]

print('Numeric features    :', num_c)
print('Categorical features:', cat_c)
print('Train:', X_train_p.shape, '| Test:', X_test_p.shape)

n_splits = int(max(2, min(CV_FOLDS, y_train.value_counts().min())))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
results = {}          # will store metrics of every model
models  = {}          # will store fitted models

## Helper: evaluation function (used for all three models)

In [ ]:
def evaluate(name, model):
    y_pred_tr = model.predict(X_train_p)
    y_pred    = model.predict(X_test_p)
    proba     = model.predict_proba(X_test_p)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average=AVG, zero_division=0)
    rec  = recall_score(y_test, y_pred, average=AVG, zero_division=0)
    f1   = f1_score(y_test, y_pred, average=AVG, zero_division=0)
    if n_classes == 2:
        roc_auc = roc_auc_score(y_test, proba[:, 1])
    else:
        roc_auc = roc_auc_score(y_test, proba, multi_class='ovr', average='macro')

    print(f'===== {name} =====')
    print(f'Train accuracy : {accuracy_score(y_train, y_pred_tr):.4f}')
    print(f'Test accuracy  : {acc:.4f}')
    print(f'Precision ({AVG}): {prec:.4f}')
    print(f'Recall    ({AVG}): {rec:.4f}')
    print(f'F1-score  ({AVG}): {f1:.4f}')
    print(f'ROC AUC        : {roc_auc:.4f}')

    cm = confusion_matrix(y_test, y_pred)
    if n_classes == 2:
        tn, fp, fn, tp = cm.ravel()
        print(f'\nTN={tn}  FP={fp}  FN={fn}  TP={tp}')
        print(f'Sensitivity (recall of positive) = {tp/(tp+fn):.4f}')
        print(f'Specificity (recall of negative) = {tn/(tn+fp):.4f}')
    print('\nClassification report:')
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title(f'Confusion Matrix - {name}')
    plt.show()

    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'ROC AUC': roc_auc}
    models[name]  = model

## Step 4: Logistic Regression
Models the probability with the sigmoid: $P(y=1) = \frac{1}{1+e^{-(b_0 + b_1x_1 + \dots)}}$. Coefficients are interpretable (odds ratio = $e^{b}$).

In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE)
log_reg.fit(X_train_p, y_train)
evaluate('Logistic Regression', log_reg)

In [ ]:
# Coefficients & odds ratios
if n_classes == 2:
    coef = pd.DataFrame({'feature': feature_names, 'coefficient': log_reg.coef_[0]})
    coef['odds_ratio'] = np.exp(coef['coefficient'])
    coef = coef.reindex(coef['coefficient'].abs().sort_values(ascending=False).index)
    print('Intercept:', log_reg.intercept_[0].round(4))
    display(coef.head(15).round(4))
    top = coef.head(15)
    plt.figure(figsize=(7, 5))
    plt.barh(top['feature'][::-1], top['coefficient'][::-1],
             color=['tomato' if v < 0 else 'seagreen' for v in top['coefficient'][::-1]])
    plt.axvline(0, color='k', lw=0.8)
    plt.title('Logistic Regression coefficients (green = pushes towards positive class)')
    plt.show()
else:
    display(pd.DataFrame(log_reg.coef_.T, index=feature_names, columns=class_names).round(3))

## Step 5: K-Nearest Neighbours (KNN)
Classifies a point by majority vote of its **k nearest** training points (distance based, so scaling is essential). Best k is chosen by cross-validation on the training set.

In [ ]:
k_range = range(1, max(2, min(K_MAX, len(X_train_p) // 2)) + 1)
cv_acc = [cross_val_score(KNeighborsClassifier(n_neighbors=k), X_train_p, y_train,
                          cv=cv, scoring='accuracy').mean() for k in k_range]
best_k = list(k_range)[int(np.argmax(cv_acc))]

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), cv_acc, marker='o')
plt.axvline(best_k, color='red', linestyle='--', label=f'best k = {best_k}')
plt.xlabel('k (number of neighbours)'); plt.ylabel('CV accuracy')
plt.title('Choosing k for KNN'); plt.xticks(list(k_range)); plt.legend()
plt.show()
print(f'Best k = {best_k}  (CV accuracy = {max(cv_acc):.4f})')

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)      # default: Euclidean distance
knn.fit(X_train_p, y_train)
evaluate(f'KNN (k={best_k})', knn)

## Step 6: Naive Bayes (Gaussian)
Uses Bayes' theorem: $P(C|x) \propto P(C)\prod_i P(x_i|C)$ with the "naive" assumption that features are **independent** given the class, and each $P(x_i|C)$ is **Gaussian**.

In [ ]:
nb = GaussianNB()
nb.fit(X_train_p, y_train)
print('Class priors P(C):', dict(zip(class_names, nb.class_prior_.round(3))))
evaluate('Naive Bayes', nb)

## Step 7: Model Comparison (test metrics + cross-validation)

In [ ]:
# Cross-validation using full pipelines (preprocessing is re-fitted inside every fold -> no leakage)
cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight=CLASS_WEIGHT, random_state=RANDOM_STATE),
    f'KNN (k={best_k})'  : KNeighborsClassifier(n_neighbors=best_k),
    'Naive Bayes'        : GaussianNB()
}
for name, mdl in cv_models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', mdl)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results[name]['CV Acc (mean)'] = scores.mean()
    results[name]['CV Acc (std)']  = scores.std()

results_df = pd.DataFrame(results).T.round(4)
display(results_df.sort_values('ROC AUC', ascending=False))
print('Best model by ROC AUC :', results_df['ROC AUC'].idxmax())
print('Best model by F1-score:', results_df['F1'].idxmax())

In [ ]:
results_df[['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC']].plot(kind='bar', figsize=(10, 5), rot=0, colormap='Set2')
plt.ylim(0, 1.05); plt.title('Model comparison (test set)'); plt.legend(loc='lower right')
plt.show()

## Step 8: ROC Curve & AUC
**ROC curve** plots TPR (recall) vs FPR at every possible threshold. **AUC** = area under it (0.5 = random guessing, 1.0 = perfect).
For multi-class data, One-vs-Rest curves are drawn per class.

In [ ]:
if n_classes == 2:
    plt.figure(figsize=(7, 6))
    for name, mdl in models.items():
        proba = mdl.predict_proba(X_test_p)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, proba)
        plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.3f})')
    plt.plot([0, 1], [0, 1], 'k--', label='Random guess (AUC = 0.5)')
    plt.xlabel('False Positive Rate (1 - Specificity)')
    plt.ylabel('True Positive Rate (Sensitivity / Recall)')
    plt.title(f"ROC Curves (positive class = '{class_names[1]}')")
    plt.legend(loc='lower right'); plt.grid(alpha=0.3)
    plt.show()
else:
    y_bin = label_binarize(y_test, classes=list(range(n_classes)))
    fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5), squeeze=False)
    for ax, (name, mdl) in zip(axes[0], models.items()):
        proba = mdl.predict_proba(X_test_p)
        all_fpr, tprs = np.linspace(0, 1, 200), []
        for i in range(n_classes):
            if y_bin[:, i].sum() == 0:
                continue
            fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
            ax.plot(fpr, tpr, lw=1.5, label=f'{class_names[i]} (AUC = {auc(fpr, tpr):.2f})')
            tprs.append(np.interp(all_fpr, fpr, tpr))
        ax.plot(all_fpr, np.mean(tprs, axis=0), 'k-', lw=2.5,
                label=f'Macro avg (AUC = {results[name]["ROC AUC"]:.2f})')
        ax.plot([0, 1], [0, 1], 'k--', lw=1)
        ax.set_title(name); ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.legend(loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

## Step 9 (Binary only): Best Threshold using Youden's J statistic

In [ ]:
if n_classes == 2:
    best_name = results_df['ROC AUC'].idxmax()
    proba = models[best_name].predict_proba(X_test_p)[:, 1]
    fpr, tpr, thr = roc_curve(y_test, proba)
    j = tpr - fpr                                  # Youden's J = sensitivity + specificity - 1
    best_thr = thr[np.argmax(j)]
    print(f'Model: {best_name}')
    print(f'Best threshold (max Youden J) = {best_thr:.3f}  (default is 0.5)')
    print(f'At this threshold -> Sensitivity = {tpr[np.argmax(j)]:.3f}, Specificity = {1 - fpr[np.argmax(j)]:.3f}')

    y_pred_thr = (proba >= best_thr).astype(int)
    print('\nAccuracy at best threshold:', round(accuracy_score(y_test, y_pred_thr), 4))
    print('F1       at best threshold:', round(f1_score(y_test, y_pred_thr), 4))

    plt.figure(figsize=(7, 5))
    plt.plot(fpr, tpr, lw=2, label='ROC')
    plt.scatter(fpr[np.argmax(j)], tpr[np.argmax(j)], color='red', s=80, zorder=5, label=f'Best threshold = {best_thr:.2f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title(f'Optimal operating point - {best_name}'); plt.legend()
    plt.show()
else:
    print('Threshold tuning is for binary classification only')

## Step 10: Save Results

In [ ]:
results_df.to_csv('classification_results.csv')
print('Saved:', os.path.join(os.getcwd(), 'classification_results.csv'))

## Theory Cheat-Sheet (for viva / written answers)

**Confusion matrix (binary):**

| | Predicted +ve | Predicted -ve |
|---|---|---|
| **Actual +ve** | TP | FN |
| **Actual -ve** | FP | TN |

| Metric | Formula | Meaning |
|---|---|---|
| Accuracy | (TP+TN)/Total | Overall correct predictions (misleading if classes are imbalanced) |
| Precision | TP/(TP+FP) | Of predicted positives, how many are truly positive |
| Recall / Sensitivity / TPR | TP/(TP+FN) | Of actual positives, how many were found |
| Specificity / TNR | TN/(TN+FP) | Of actual negatives, how many were found |
| F1-score | 2PR/(P+R) | Harmonic mean of precision and recall |
| FPR | FP/(FP+TN) = 1 - Specificity | x-axis of the ROC curve |
| AUC | area under ROC | 0.5 = random, 0.7-0.8 acceptable, 0.8-0.9 good, >0.9 excellent |

| Model | Key idea | Assumptions / notes |
|---|---|---|
| **Logistic Regression** | Linear decision boundary, sigmoid gives probability | Linear relation between features and log-odds; interpretable coefficients |
| **KNN** | Majority vote of k nearest points | Lazy learner (no training); needs **scaling**; small k = overfit, large k = underfit |
| **Naive Bayes** | Bayes' theorem + feature independence | Fast, works with small data; independence assumption rarely true but still works well |

**ROC reading tips:** a curve closer to the top-left corner is better; the diagonal is a random classifier; the model with the larger AUC ranks positives above negatives more often. Use **recall/F1 (not accuracy)** when classes are imbalanced or when missing a positive is costly.